In [8]:
import pandas as pd

# Load datasets
customers_df = pd.read_excel("https://cdn.enqurious.com/others/46de2415-dc50-46ed-8133-cccc2f034680_globalmart.xlsx", sheet_name='customers') # Add the path

orders_df = pd.read_excel("https://cdn.enqurious.com/others/46de2415-dc50-46ed-8133-cccc2f034680_globalmart.xlsx", sheet_name='orders') # Add the path

trans = pd.read_excel("https://cdn.enqurious.com/others/46de2415-dc50-46ed-8133-cccc2f034680_globalmart.xlsx", sheet_name='transactions')
# =====================================================================
# STEP 1: Combine Orders & Transaction Data to gather dates and amounts
# =====================================================================
# orders_df gives us 'order_purchase_date' and 'customer_id'
# trans gives us the financial column 'sales_amt'
order_trans_df = orders_df.merge(trans, on='order_id', how='inner')

# =====================================================================
# STEP 2: Filter Data for the Last 6 Months
# =====================================================================
# Ensure the purchase date column is parsed into proper datetime objects
order_trans_df['order_purchase_date'] = pd.to_datetime(order_trans_df['order_purchase_date'])

# Establish the 6-month cutoff date relative to the latest transaction in the dataset
cutoff_date = order_trans_df['order_purchase_date'].max() - pd.DateOffset(months=6)
recent_orders = order_trans_df[order_trans_df['order_purchase_date'] >= cutoff_date]

# =====================================================================
# STEP 3: Aggregate Spending and Order Frequencies per Customer
# =====================================================================
customer_summary = recent_orders.groupby('customer_id').agg(
    total_spending=('sales_amt', 'sum'),
    order_count=('order_id', 'count')
).reset_index()

# =====================================================================
# STEP 4: Define Conditional Segmentation and Tier Logic Rules
# =====================================================================
def assign_tier(row):
    if row['total_spending'] < 500:
        return 'Silver'
    elif row['total_spending'] <= 2000:
        return 'Gold'
    else:
        return 'Platinum'

def assign_discount(row):
    if row['tier'] == 'Silver':
        return 4 if row['order_count'] >= 10 else 2
    elif row['tier'] == 'Gold':
        return 8 if row['order_count'] >= 10 else 6
    else: # Platinum Tier
        return 15 if row['order_count'] >= 10 else 10

# =====================================================================
# STEP 5: Run Segmentations and Generate the Loyalty Report
# =====================================================================
# Apply the conditional rule matrices to calculate tiers and discounts
customer_summary['tier'] = customer_summary.apply(assign_tier, axis=1)
customer_summary['discount_applicable'] = customer_summary.apply(assign_discount, axis=1)

# Left join with customers_df to cleanly match descriptive customer names
loyalty_report = customer_summary.merge(
    customers_df[['customer_id', 'customer_name']],
    on='customer_id',
    how='left'
)

# Restructure final schema to match the requested output requirements
loyalty_report = loyalty_report[[
    'customer_id', 'customer_name', 'total_spending',
    'order_count', 'tier', 'discount_applicable'
]]

# Preview the finished Customer Loyalty Tier Report
loyalty_report.head()

,customer_id,customer_name,total_spending,order_count,tier,discount_applicable
0,AA-10315,Alex Avila,2637.518,2,Platinum,10
1,AA-10375,Allen Armold,29.320,3,Silver,2
2,AA-10480,Andrew Allen,2235.244,6,Platinum,10
3,AA-10645,Anna Andreadi,230.895,7,Silver,2
4,AB-10015,Aaron Bergman,77.144,5,Silver,2


In [20]:
loyalty_report.shape


(731, 6)

In [10]:
cutoff_date


Timestamp('2018-02-28 13:07:00')

In [21]:
recent_orders.shape

(4143, 16)

In [12]:
recent_orders

,order_id,customer_id,ship_mode,vendor_id,order_status,order_purchase_date,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,id,product_id,sales_amt,qty,discount,profit_amt
10,CA-2014-103100,AB-10105,First Class,VEN02,delivered,2018-08-15 09:38:00,2018-08-17 03:10,2018-08-17 15:40,2018-08-22 21:13,2018-09-12,4284,OFF-ST-10001580,35.952,3,0.2,3.5952
11,CA-2014-103100,AB-10105,First Class,VEN02,delivered,2018-08-15 09:38:00,2018-08-17 03:10,2018-08-17 15:40,2018-08-22 21:13,2018-09-12,4283,OFF-AR-10001573,6.990,3,0.0,2.0271
12,CA-2014-103317,DM-13525,First Class,VEN01,delivered,2018-08-06 08:44:00,2018-08-06 09:04,2018-08-07 16:11,2018-08-14 22:18,2018-08-21,5439,OFF-ST-10003455,46.530,3,0.0,12.0978
13,CA-2014-103317,DM-13525,First Class,VEN01,delivered,2018-08-06 08:44:00,2018-08-06 09:04,2018-08-07 16:11,2018-08-14 22:18,2018-08-21,5437,OFF-AR-10001427,11.960,2,0.0,3.1096
14,CA-2014-103317,DM-13525,First Class,VEN01,delivered,2018-08-06 08:44:00,2018-08-06 09:04,2018-08-07 16:11,2018-08-14 22:18,2018-08-21,5438,FUR-TA-10004607,517.405,5,0.3,-81.3065
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9819,US-2017-166233,MO-17950,Standard Class,VEN01,delivered,2018-03-02 19:13:00,2018-03-03 02:35,2018-03-27 15:51,2018-04-03 20:08,2018-03-29,7870,OFF-AR-10003903,32.776,1,0.2,2.4582
9820,US-2017-166611,CK-12760,Standard Class,VEN01,delivered,2018-03-15 20:23:00,2018-03-16 20:31,2018-03-20 23:42,2018-03-29 19:22,2018-04-06,1880,OFF-ST-10002301,20.340,1,0.0,0.2034
9825,US-2017-167570,EG-13900,Standard Class,VEN01,delivered,2018-08-02 12:32:00,2018-08-02 13:23,2018-08-03 12:20,2018-08-07 21:04,2018-08-17,8109,OFF-ST-10001809,269.490,3,0.0,5.3898
9829,US-2017-169502,MG-17650,Standard Class,VEN05,delivered,2018-04-08 23:05:00,2018-04-09 08:30,2018-04-10 17:03,2018-04-21 16:21,2018-05-02,4599,OFF-SU-10000157,835.170,7,0.0,16.7034


In [13]:
# Load datasets
customers_df = pd.read_excel("https://cdn.enqurious.com/others/46de2415-dc50-46ed-8133-cccc2f034680_globalmart.xlsx", sheet_name='customers') # Add the path

orders_df = pd.read_excel("https://cdn.enqurious.com/others/46de2415-dc50-46ed-8133-cccc2f034680_globalmart.xlsx", sheet_name='orders') # Add the path

trans = pd.read_excel("https://cdn.enqurious.com/others/46de2415-dc50-46ed-8133-cccc2f034680_globalmart.xlsx", sheet_name='transactions')

In [14]:
# new_orders_df is already loaded
# loyalty_report is available from the previous step

# This dataset contains the upcoming orders that need promotional pricing applied
new_orders_url = "https://cdn.enqurious.com/documents/517cad90-cfb3-48fe-975e-256d942412ac_neworders.csv"

In [15]:
cities = ["Delhi", "Mumbai", "Chennai"]

# Indexing (same as strings)
cities[0]           # 'Delhi'
cities[-1]          # 'Chennai'
cities[0:2]          # ['Delhi', 'Mumbai']

# Adding
cities.append("Kolkata")    # add to end
cities.insert(1, "Pune")  # insert at index 1

# Removing
cities.remove("Pune")       # remove by value
cities.pop()              # remove last item
cities.pop(0)             # remove at index 0

# Combining
more = ["Bengaluru", "Jaipur"]
cities.extend(more)       # add all items

# Sorting
cities.sort()              # A-Z in place
cities.sort(reverse=True) # Z-A
len(cities)               # count of items

4

In [16]:
for city in cities:
    print(city)

Mumbai
Jaipur
Chennai
Bengaluru


In [17]:
for i, city in enumerate(cities):
    print(i, city)

0 Mumbai
1 Jaipur
2 Chennai
3 Bengaluru


In [18]:
import pandas as pd
import numpy as np

# =====================================================================
# STEP 1: Load the New Orders Dataset
# =====================================================================
# This dataset contains the upcoming orders that need promotional pricing applied
new_orders_url = "https://cdn.enqurious.com/documents/517cad90-cfb3-48fe-975e-256d942412ac_neworders.csv"
new_orders_df = pd.read_csv(new_orders_url)

# Clean column names to lowercase to prevent capitalization mismatches (e.g., Order_id vs order_id)
new_orders_df.columns = new_orders_df.columns.str.lower()

# =====================================================================
# STEP 2: Merge New Orders with the Loyalty Report
# =====================================================================
# We use a LEFT join so that all upcoming orders are retained,
# even if a customer is completely new and has no historical tier data.
pricing_report = new_orders_df.merge(
    loyalty_report[['customer_id', 'customer_name', 'tier', 'discount_applicable']],
    on='customer_id',
    how='left'
)

# =====================================================================
# STEP 3: Handle New/Unmatched Customers (Data Cleaning)
# =====================================================================
# New customers without history will inherit NaN values for their discount.
# We explicitly fill these missing values with 0% so the pricing math runs perfectly.
pricing_report['discount_applicable'] = pricing_report['discount_applicable'].fillna(0)
pricing_report['tier'] = pricing_report['tier'].fillna('No Tier')
# Fill missing customer names with 'Unknown Customer'
pricing_report['customer_name'] = pricing_report['customer_name'].fillna('Unknown Customer')

# =====================================================================
# STEP 4: Calculate the Final Discounted Price
# =====================================================================
# Formula: Original Price × (1 − Discount Applicable / 100)
pricing_report['discounted_price'] = (
    pricing_report['original_price'] * (1 - pricing_report['discount_applicable'] / 100)
)

# Optional: Round the currency calculations to 2 decimal places for neatness
pricing_report['discounted_price'] = pricing_report['discounted_price'].round(2)

# =====================================================================
# STEP 5: Reorder and Select Specified Output Columns
# =====================================================================
pricing_report = pricing_report[[
    'order_id', 'customer_id', 'customer_name',
    'original_price', 'discount_applicable', 'discounted_price'
]]

# Display the first 5 rows of your final Discounted Pricing Report
pricing_report.head()

,order_id,customer_id,customer_name,original_price,discount_applicable,discounted_price
0,CA-2014-103392,AB-10105,Adrian Barton,55.462,8.0,51.03
1,CA-2014-103254,DM-13525,Don Miller,66.374,6.0,62.39
2,CA-2014-103445,RT-13456,NaN,588.678,0.0,588.68
3,CA-2014-103136,JH-10250,NaN,224.731,0.0,224.73
4,CA-2014-103283,AB-10105,Adrian Barton,578.572,8.0,532.29


In [22]:
recent_orders.shape


(4143, 16)

In [19]:
# Convert the purchase date to datetime
orders_df['order_purchase_date'] = pd.to_datetime(orders_df['order_purchase_date'])

# Find the latest date in the dataset
latest_date = orders_df['order_purchase_date'].max()

# Calculate the date 6 months before the latest date
six_months_ago = latest_date - pd.DateOffset(months=6)

# Filter orders from the last 6 months
last_6_months_orders = orders_df[
    orders_df['order_purchase_date'] >= six_months_ago
]

# Count remaining orders
print("Orders remaining:", len(last_6_months_orders))

Orders remaining: 2041


In [23]:
latest_date

Timestamp('2018-08-30 13:07:00')

In [24]:
six_months_ago

Timestamp('2018-02-28 13:07:00')

In [26]:
last_6_months_orders.shape


(2041, 10)